# Card 2 – Incident Future Trends: Best Model Selection

**For each template:**
1. Run ARIMA, SARIMAX, Exponential Smoothing, Prophet **in parallel** with hyperparameter tuning
2. Print comparison table — pick winner by MAE
3. Show only the winning model's Plotly chart

**Templates:**
1. Overall Incident Count Forecast
2. Total Injuries Forecast
3. Operator Injuries Forecast
4. Rider Injuries Forecast
5. Monthly Seasonality Forecast
6. Year-over-Year + Forecast Year

In [22]:
import itertools
import warnings
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from prophet import Prophet
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.statespace.sarimax import SARIMAX

warnings.filterwarnings('ignore')
pio.renderers.default = 'browser'

CTA_BLUE       = '#0066B3'
FORECAST_COLOR = '#9c27b0'
CONF_COLOR     = 'rgba(156,39,176,0.15)'
TEST_COLOR     = '#fb8c00'

print('✅ Libraries loaded')

✅ Libraries loaded


In [23]:
CSV_PATH = r'C:\Users\azeez\PROJECTS\CTA-MAP-ASSAULT\Cleaned_CTA_Bus_Data.csv'
df = pd.read_csv(CSV_PATH, low_memory=False)

def parse_cta_date(d):
    if pd.isna(d): return pd.NaT
    try:    return pd.to_datetime(d, format='%Y %B %d')
    except: return pd.to_datetime(d, errors='coerce')

df['parsed_date'] = df['Event Date'].apply(parse_cta_date)
df = df[df['parsed_date'].notna()].copy()
df['year_month'] = df['parsed_date'].dt.to_period('M')

# injury columns
df['Total Injuries']                      = pd.to_numeric(df.get('Total Injuries', 0),                      errors='coerce').fillna(0)
df['Transit Vehicle Operator Injuries']   = pd.to_numeric(df.get('Transit Vehicle Operator Injuries', 0),   errors='coerce').fillna(0)
df['Transit Vehicle Rider Injuries']      = pd.to_numeric(df.get('Transit Vehicle Rider Injuries', 0),      errors='coerce').fillna(0)

print(f'Rows: {len(df):,}  |  Date range: {df["parsed_date"].min().date()} → {df["parsed_date"].max().date()}')
print(f'Months: {df["year_month"].nunique()}')

Rows: 3,749  |  Date range: 2014-01-02 → 2025-04-28
Months: 136


## Model Training Functions (each with hyperparameter tuning)

In [24]:
def _metrics(actual, predicted):
    actual    = np.array(actual, dtype=float)
    predicted = np.array(predicted, dtype=float)
    mae  = mean_absolute_error(actual, predicted)
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    mape = np.mean(np.abs((actual - predicted) / np.where(actual == 0, 1, actual))) * 100
    return mae, rmse, mape

def _forecast_dates(series, periods):
    return pd.date_range(start=series.index[-1] + pd.DateOffset(months=1), periods=periods, freq='MS')

def _result(name, series, train, test, test_fc, fc_vals, lower, upper, fd, mae, mape):
    return dict(
        model_name=name, historical=series,
        train=train, test=test,
        test_forecast=pd.Series(np.array(test_fc), index=test.index),
        forecast=np.maximum(np.array(fc_vals), 0),
        forecast_lower=np.array(lower),
        forecast_upper=np.array(upper),
        forecast_dates=fd, mae=mae, mape=mape,
    )

# ── ARIMA (grid search p,d,q) ────────────────────────────────────────────────
def train_arima_tuned(series, forecast_periods=12):
    if len(series) < 18: return None
    train, test = series[:-6], series[-6:]
    best_aic, best_p, best_m = np.inf, None, None
    for p, d, q in itertools.product(range(3), range(2), range(3)):
        try:
            m = ARIMA(train, order=(p,d,q)).fit()
            if m.aic < best_aic:
                best_aic, best_p, best_m = m.aic, (p,d,q), m
        except: continue
    if best_m is None: return None
    test_fc = best_m.forecast(steps=6)
    mae, _, mape = _metrics(test, test_fc)
    full = ARIMA(series, order=best_p).fit()
    fc   = full.get_forecast(steps=forecast_periods)
    conf = fc.conf_int()
    fd   = _forecast_dates(series, forecast_periods)
    return _result(f'ARIMA{best_p}', series, train, test, test_fc,
                   fc.predicted_mean.values, conf.iloc[:,0].values, conf.iloc[:,1].values, fd, mae, mape)

# ── SARIMAX (grid search (p,d,q)(P,1,Q,12)) ─────────────────────────────────
def train_sarimax_tuned(series, forecast_periods=12):
    if len(series) < 24: return None
    train, test = series[:-6], series[-6:]
    best_aic, best_p, best_m = np.inf, None, None
    for p, d, q in itertools.product([0,1], [0,1], [0,1]):
        for P, Q in itertools.product([0,1], [0,1]):
            try:
                m = SARIMAX(train, order=(p,d,q), seasonal_order=(P,1,Q,12)).fit(disp=False, maxiter=50)
                if m.aic < best_aic:
                    best_aic, best_p, best_m = m.aic, ((p,d,q),(P,1,Q,12)), m
            except: continue
    if best_m is None: return None
    test_fc = best_m.forecast(steps=6)
    mae, _, mape = _metrics(test, test_fc)
    full = SARIMAX(series, order=best_p[0], seasonal_order=best_p[1]).fit(disp=False)
    fc   = full.get_forecast(steps=forecast_periods)
    conf = fc.conf_int()
    fd   = _forecast_dates(series, forecast_periods)
    return _result(f'SARIMAX{best_p[0]}×{best_p[1]}', series, train, test, test_fc,
                   fc.predicted_mean.values, conf.iloc[:,0].values, conf.iloc[:,1].values, fd, mae, mape)

# ── Exponential Smoothing (grid search trend × seasonal) ─────────────────────
def train_ets_tuned(series, forecast_periods=12):
    if len(series) < 18: return None
    train, test = series[:-6], series[-6:]
    best_aic, best_p, best_m = np.inf, None, None
    for trend in ['add', 'mul', None]:
        for seasonal in ['add', 'mul', None]:
            sp = 12 if seasonal else None
            try:
                m = ExponentialSmoothing(train, trend=trend, seasonal=seasonal, seasonal_periods=sp).fit()
                if m.aic < best_aic:
                    best_aic, best_p, best_m = m.aic, (trend, seasonal, sp), m
            except: continue
    if best_m is None: return None
    test_fc = best_m.forecast(steps=6)
    mae, _, mape = _metrics(test, test_fc)
    full    = ExponentialSmoothing(series, trend=best_p[0], seasonal=best_p[1], seasonal_periods=best_p[2]).fit()
    fc_vals = full.forecast(steps=forecast_periods)
    fd      = _forecast_dates(series, forecast_periods)
    # approximate 80% CI from residual std
    std = np.std(full.resid) * 1.28
    return _result(f'ETS(t={best_p[0]},s={best_p[1]})', series, train, test, test_fc,
                   fc_vals.values, fc_vals.values - std, fc_vals.values + std, fd, mae, mape)

# ── Prophet (grid search changepoint × seasonality scales) ───────────────────
def train_prophet_tuned(series, forecast_periods=12):
    if len(series) < 18: return None
    pdf        = series.reset_index()
    pdf.columns = ['ds', 'y']
    train_df, test_df = pdf.iloc[:-6], pdf.iloc[-6:]
    best_mae, best_p = np.inf, None
    for cp in [0.001, 0.01, 0.1, 0.5]:
        for sp in [0.01, 0.1, 1.0, 10.0]:
            try:
                m = Prophet(yearly_seasonality=True, weekly_seasonality=False,
                            daily_seasonality=False, changepoint_prior_scale=cp,
                            seasonality_prior_scale=sp, interval_width=0.80)
                m.fit(train_df)
                pred = m.predict(m.make_future_dataframe(periods=6, freq='MS'))
                mae_val = mean_absolute_error(test_df['y'].values, pred.tail(6)['yhat'].values)
                if mae_val < best_mae:
                    best_mae, best_p = mae_val, (cp, sp)
            except: continue
    if best_p is None: return None
    final = Prophet(yearly_seasonality=True, weekly_seasonality=False,
                    daily_seasonality=False, changepoint_prior_scale=best_p[0],
                    seasonality_prior_scale=best_p[1], interval_width=0.80)
    final.fit(pdf)
    fut  = final.make_future_dataframe(periods=forecast_periods, freq='MS')
    pred = final.predict(fut)
    n    = len(pdf)
    test_fc_vals = pred.iloc[n-6:n]['yhat'].values
    fc_rows      = pred.tail(forecast_periods)
    mae, _, mape = _metrics(series.iloc[-6:].values, test_fc_vals)
    fd           = _forecast_dates(series, forecast_periods)
    return _result(f'Prophet(cp={best_p[0]},sp={best_p[1]})', series,
                   series.iloc[:-6], series.iloc[-6:], test_fc_vals,
                   fc_rows['yhat'].values, fc_rows['yhat_lower'].values,
                   fc_rows['yhat_upper'].values, fd, mae, mape)

print('✅ All 4 model functions defined')

✅ All 4 model functions defined


## Parallel Runner + Chart Builder

In [25]:
def run_best_model(series, template_name, forecast_periods=12):
    """Run all 4 models in parallel with tuning. Print comparison. Return best result."""
    print(f'\n{"="*60}')
    print(f'  {template_name}')
    print(f'{"="*60}')

    model_funcs = {
        'ARIMA':         train_arima_tuned,
        'SARIMAX':       train_sarimax_tuned,
        'Exp Smoothing': train_ets_tuned,
        'Prophet':       train_prophet_tuned,
    }

    results = {}
    with ThreadPoolExecutor(max_workers=4) as ex:
        futures = {ex.submit(fn, series, forecast_periods): name for name, fn in model_funcs.items()}
        for f in as_completed(futures):
            name = futures[f]
            try:
                r = f.result()
                if r:
                    results[name] = r
            except Exception as e:
                print(f'  ❌ {name}: {e}')

    if not results:
        raise ValueError('All models failed')

    # comparison table
    rows = sorted(results.values(), key=lambda r: r['mae'])
    best = rows[0]
    print(f'\n  {"Model":<35} {"MAE":>6}  {"MAPE":>7}')
    print(f'  {"─"*52}')
    for r in rows:
        star = '  ⭐ BEST' if r is best else ''
        print(f'  {r["model_name"]:<35} {r["mae"]:>6.2f}  {r["mape"]:>6.1f}%{star}')

    return best, pd.DataFrame([{'Model': r['model_name'], 'MAE': round(r['mae'],2), 'MAPE%': round(r['mape'],1)} for r in rows])


def build_forecast_chart(result, title, y_label='Incidents', forecast_periods=12):
    """Plotly figure: historical + validation + forecast + confidence band."""
    hist     = result['historical']
    fc_dates = result['forecast_dates']
    train_n  = len(result['train'])

    fig = go.Figure()

    # confidence band
    fig.add_trace(go.Scatter(
        x=list(fc_dates) + list(fc_dates[::-1]),
        y=list(np.maximum(result['forecast_upper'], 0)) + list(np.maximum(result['forecast_lower'], 0)[::-1]),
        fill='toself', fillcolor=CONF_COLOR,
        line=dict(color='rgba(0,0,0,0)'), name='80% Confidence', hoverinfo='skip',
    ))
    # historical train
    fig.add_trace(go.Scatter(
        x=hist.index[:train_n], y=hist.values[:train_n],
        mode='lines+markers', name='Historical',
        line=dict(color=CTA_BLUE, width=2), marker=dict(size=4),
    ))
    # actual validation period
    fig.add_trace(go.Scatter(
        x=hist.index[train_n:], y=hist.values[train_n:],
        mode='lines+markers', name='Actual (validation)',
        line=dict(color=CTA_BLUE, width=2, dash='dot'), marker=dict(size=5, symbol='circle-open'),
    ))
    # model validation
    fig.add_trace(go.Scatter(
        x=result['test_forecast'].index, y=result['test_forecast'].values,
        mode='lines', name='Model validation',
        line=dict(color=TEST_COLOR, width=2, dash='dash'),
    ))
    # forecast
    fig.add_trace(go.Scatter(
        x=fc_dates, y=result['forecast'],
        mode='lines+markers', name=f'{result["model_name"]} forecast',
        line=dict(color=FORECAST_COLOR, width=3, dash='dash'),
        marker=dict(size=6, symbol='diamond'),
    ))

    # Today line
    fig.add_vline(x=hist.index[-1].timestamp()*1000, line_dash='dot', line_color='gray',
                  annotation_text='Today', annotation_position='top right', annotation_font_size=11)

    # metrics box
    hist_avg = hist.mean()
    fc_avg   = result['forecast'].mean()
    pct      = (fc_avg - hist_avg) / hist_avg * 100 if hist_avg else 0
    arrow    = '▲' if pct > 0 else '▼'
    clr      = 'red' if pct > 5 else ('green' if pct < -5 else 'gray')
    fig.add_annotation(
        xref='paper', yref='paper', x=0.01, y=0.99, showarrow=False,
        xanchor='left', yanchor='top', font=dict(size=12),
        bgcolor='rgba(255,255,255,0.88)', bordercolor='lightgray', borderwidth=1,
        text=(f"<b>Model:</b> {result['model_name']} &nbsp;&nbsp;"
              f"<b>MAE:</b> {result['mae']:.1f} &nbsp;&nbsp;"
              f"<b>MAPE:</b> {result['mape']:.1f}% &nbsp;&nbsp;"
              f"<b><span style='color:{clr}'>{arrow}{abs(pct):.1f}%</span></b> vs hist avg"),
    )

    fig.update_layout(
        title=dict(text=title, font=dict(size=18, color='#1a202c')),
        xaxis=dict(title='Date', showgrid=True, gridcolor='rgba(0,0,0,0.06)'),
        yaxis=dict(title=y_label, showgrid=True, gridcolor='rgba(0,0,0,0.06)'),
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
        plot_bgcolor='white', paper_bgcolor='white',
        height=430, margin=dict(l=60, r=30, t=90, b=60), hovermode='x unified',
    )
    return fig

print('✅ run_best_model() and build_forecast_chart() defined')

✅ run_best_model() and build_forecast_chart() defined


---
## Template 1 — Overall Incident Count Forecast

In [26]:
monthly_count = df.groupby('year_month').size().to_timestamp()

best_count, comp1 = run_best_model(monthly_count, 'Template 1 — Overall Incident Count', forecast_periods=12)

fig1 = build_forecast_chart(best_count, 'Incident Count Forecast — Next 12 Months', 'Incidents/Month')
fig1.show()


  Template 1 — Overall Incident Count


15:10:26 - cmdstanpy - INFO - Chain [1] start processing
15:10:26 - cmdstanpy - INFO - Chain [1] done processing
15:10:26 - cmdstanpy - ERROR - Chain [1] error: code '1' Operation not permitted
Optimization terminated abnormally. Falling back to Newton.
15:10:26 - cmdstanpy - INFO - Chain [1] start processing
15:10:27 - cmdstanpy - INFO - Chain [1] done processing
15:10:27 - cmdstanpy - INFO - Chain [1] start processing
15:10:27 - cmdstanpy - INFO - Chain [1] done processing
15:10:27 - cmdstanpy - ERROR - Chain [1] error: code '1' Operation not permitted
Optimization terminated abnormally. Falling back to Newton.
15:10:27 - cmdstanpy - INFO - Chain [1] start processing
15:10:28 - cmdstanpy - INFO - Chain [1] done processing
15:10:29 - cmdstanpy - INFO - Chain [1] start processing
15:10:29 - cmdstanpy - INFO - Chain [1] done processing
15:10:29 - cmdstanpy - ERROR - Chain [1] error: code '1' Operation not permitted
Optimization terminated abnormally. Falling back to Newton.
15:10:29 - c


  Model                                  MAE     MAPE
  ────────────────────────────────────────────────────
  SARIMAX(0, 1, 1)×(0, 1, 1, 12)        3.14    11.6%  ⭐ BEST
  Prophet(cp=0.1,sp=0.01)               3.40    12.2%
  ARIMA(1, 1, 1)                        4.15    15.7%
  ETS(t=None,s=None)                    4.68    17.9%


---
## Template 2 — Total Injuries Forecast

In [27]:
monthly_injuries = df.groupby('year_month')['Total Injuries'].sum().to_timestamp()
monthly_injuries = monthly_injuries[monthly_injuries.index >= '2016-01-01']

best_injuries, comp2 = run_best_model(monthly_injuries, 'Template 2 — Total Injuries', forecast_periods=12)

fig2 = build_forecast_chart(best_injuries, 'Total Injuries Forecast — Next 12 Months', 'Injuries/Month')
fig2.show()


  Template 2 — Total Injuries


15:10:33 - cmdstanpy - INFO - Chain [1] start processing
15:10:33 - cmdstanpy - INFO - Chain [1] done processing
15:10:34 - cmdstanpy - ERROR - Chain [1] error: code '1' Operation not permitted
Optimization terminated abnormally. Falling back to Newton.
15:10:35 - cmdstanpy - INFO - Chain [1] start processing
15:10:35 - cmdstanpy - INFO - Chain [1] done processing
15:10:36 - cmdstanpy - INFO - Chain [1] start processing
15:10:37 - cmdstanpy - INFO - Chain [1] done processing
15:10:37 - cmdstanpy - ERROR - Chain [1] error: code '1' Operation not permitted
Optimization terminated abnormally. Falling back to Newton.
15:10:37 - cmdstanpy - INFO - Chain [1] start processing
15:10:37 - cmdstanpy - INFO - Chain [1] done processing
15:10:38 - cmdstanpy - INFO - Chain [1] start processing
15:10:38 - cmdstanpy - INFO - Chain [1] done processing
15:10:38 - cmdstanpy - ERROR - Chain [1] error: code '1' Operation not permitted
Optimization terminated abnormally. Falling back to Newton.
15:10:38 - c


  Model                                  MAE     MAPE
  ────────────────────────────────────────────────────
  SARIMAX(1, 1, 1)×(0, 1, 1, 12)        5.83    15.8%  ⭐ BEST
  Prophet(cp=0.1,sp=0.01)               5.98    16.5%
  ARIMA(0, 1, 2)                        7.78    21.3%
  ETS(t=None,s=None)                   10.35    27.5%


---
## Template 3 — Operator Injuries Forecast

In [28]:
monthly_op = df.groupby('year_month')['Transit Vehicle Operator Injuries'].sum().to_timestamp()
monthly_op = monthly_op[monthly_op.index >= '2016-01-01']

best_op, comp3 = run_best_model(monthly_op, 'Template 3 — Operator Injuries', forecast_periods=12)

fig3 = build_forecast_chart(best_op, 'Operator Injuries Forecast — Next 12 Months', 'Injuries/Month')
fig3.show()


  Template 3 — Operator Injuries


15:10:51 - cmdstanpy - INFO - Chain [1] start processing
15:10:51 - cmdstanpy - INFO - Chain [1] done processing
15:10:52 - cmdstanpy - ERROR - Chain [1] error: code '1' Operation not permitted
Optimization terminated abnormally. Falling back to Newton.
15:10:52 - cmdstanpy - INFO - Chain [1] start processing
15:10:52 - cmdstanpy - INFO - Chain [1] done processing
15:10:54 - cmdstanpy - INFO - Chain [1] start processing
15:10:54 - cmdstanpy - INFO - Chain [1] done processing
15:10:54 - cmdstanpy - ERROR - Chain [1] error: code '1' Operation not permitted
Optimization terminated abnormally. Falling back to Newton.
15:10:55 - cmdstanpy - INFO - Chain [1] start processing
15:10:55 - cmdstanpy - INFO - Chain [1] done processing
15:10:55 - cmdstanpy - INFO - Chain [1] start processing
15:10:56 - cmdstanpy - INFO - Chain [1] done processing
15:10:56 - cmdstanpy - ERROR - Chain [1] error: code '1' Operation not permitted
Optimization terminated abnormally. Falling back to Newton.
15:10:56 - c


  Model                                  MAE     MAPE
  ────────────────────────────────────────────────────
  ARIMA(0, 1, 1)                        3.08    18.5%  ⭐ BEST
  Prophet(cp=0.5,sp=0.01)               3.21    20.2%
  SARIMAX(0, 1, 1)×(0, 1, 1, 12)        3.23    17.4%
  ETS(t=mul,s=None)                     3.67    20.7%


---
## Template 4 — Rider Injuries Forecast

In [29]:
monthly_rider = df.groupby('year_month')['Transit Vehicle Rider Injuries'].sum().to_timestamp()
monthly_rider = monthly_rider[monthly_rider.index >= '2016-01-01']

best_rider, comp4 = run_best_model(monthly_rider, 'Template 4 — Rider Injuries', forecast_periods=12)

fig4 = build_forecast_chart(best_rider, 'Rider Injuries Forecast — Next 12 Months', 'Injuries/Month')
fig4.show()


  Template 4 — Rider Injuries


15:11:07 - cmdstanpy - INFO - Chain [1] start processing
15:11:07 - cmdstanpy - INFO - Chain [1] done processing
15:11:09 - cmdstanpy - INFO - Chain [1] start processing
15:11:09 - cmdstanpy - INFO - Chain [1] done processing
15:11:12 - cmdstanpy - INFO - Chain [1] start processing
15:11:12 - cmdstanpy - INFO - Chain [1] done processing
15:11:13 - cmdstanpy - INFO - Chain [1] start processing
15:11:13 - cmdstanpy - INFO - Chain [1] done processing
15:11:14 - cmdstanpy - INFO - Chain [1] start processing
15:11:14 - cmdstanpy - INFO - Chain [1] done processing
15:11:14 - cmdstanpy - INFO - Chain [1] start processing
15:11:14 - cmdstanpy - INFO - Chain [1] done processing
15:11:15 - cmdstanpy - INFO - Chain [1] start processing
15:11:15 - cmdstanpy - INFO - Chain [1] done processing
15:11:16 - cmdstanpy - INFO - Chain [1] start processing
15:11:16 - cmdstanpy - INFO - Chain [1] done processing
15:11:16 - cmdstanpy - INFO - Chain [1] start processing
15:11:17 - cmdstanpy - INFO - Chain [1]


  Model                                  MAE     MAPE
  ────────────────────────────────────────────────────
  Prophet(cp=0.1,sp=10.0)               3.22    15.3%  ⭐ BEST
  SARIMAX(1, 1, 1)×(0, 1, 1, 12)        5.85    32.3%
  ARIMA(0, 1, 2)                        6.30    38.9%
  ETS(t=None,s=None)                    7.00    43.5%


---
## Template 5 — Monthly Seasonality Forecast
Shows average incidents per calendar month historically, then forecasts next 12 months projected by month.

In [30]:
MONTH_NAMES = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

# Historical avg per calendar month
df['month_num'] = df['parsed_date'].dt.month
hist_seasonal   = df.groupby('month_num').size() / df['parsed_date'].dt.year.nunique()

# Forecast avg per calendar month (from best count model)
fc_df = pd.DataFrame({
    'month': pd.DatetimeIndex(best_count['forecast_dates']).month,
    'fc':    best_count['forecast'],
})
fc_seasonal = fc_df.groupby('month')['fc'].mean()

fig5 = go.Figure()
fig5.add_trace(go.Bar(
    x=[MONTH_NAMES[i-1] for i in hist_seasonal.index], y=hist_seasonal.values,
    name='Historical avg/month', marker_color=CTA_BLUE,
))
fig5.add_trace(go.Bar(
    x=[MONTH_NAMES[i-1] for i in fc_seasonal.index], y=fc_seasonal.values,
    name=f'Forecast ({best_count["model_name"]})', marker_color=FORECAST_COLOR, opacity=0.85,
))
fig5.update_layout(
    title=f'Monthly Seasonal Pattern — Historical vs Forecast ({best_count["model_name"]})',
    barmode='group', xaxis_title='Month', yaxis_title='Avg Incidents',
    plot_bgcolor='white', height=400, legend=dict(orientation='h', y=1.12),
)
fig5.show()

---
## Template 6 — Year-over-Year + Forecast Year
Shows each historical year as a line, plus a forecast year line.

In [31]:
df['month_num'] = df['parsed_date'].dt.month
df['year']      = df['parsed_date'].dt.year
pivot = df.groupby(['year','month_num']).size().unstack(level=0, fill_value=0)

fig6 = go.Figure()
years = sorted(pivot.columns)
for i, yr in enumerate(years):
    alpha = 0.2 + 0.8 * (i / max(len(years)-1, 1))
    fig6.add_trace(go.Scatter(
        x=[MONTH_NAMES[m-1] for m in pivot.index], y=pivot[yr].values,
        mode='lines', name=str(yr),
        line=dict(color=f'rgba(0,102,179,{alpha:.2f})', width=2 if yr == years[-1] else 1),
    ))

# forecast year from best model
fc_by_month = fc_df.groupby('month')['fc'].mean().reindex(range(1,13), fill_value=np.nan)
fig6.add_trace(go.Scatter(
    x=[MONTH_NAMES[m-1] for m in range(1,13)],
    y=np.maximum(fc_by_month.values, 0),
    mode='lines+markers', name=f'Forecast ({best_count["model_name"]})',
    line=dict(color=FORECAST_COLOR, width=3, dash='dash'),
    marker=dict(size=7, symbol='diamond'),
))
fig6.update_layout(
    title=f'Year-over-Year + Forecast ({best_count["model_name"]})',
    xaxis_title='Month', yaxis_title='Incidents',
    plot_bgcolor='white', height=440, hovermode='x unified',
    legend=dict(orientation='v', x=1.01, y=1),
)
fig6.show()

---
## Summary of all 6 templates side by side

In [32]:
print('='*65)
print('  FINAL SUMMARY — Best Model per Template')
print('='*65)

summary = pd.DataFrame([
    {'Template': 'T1 – Incident Count',    'Best Model': best_count['model_name'],    'MAE': best_count['mae'],    'MAPE%': best_count['mape']},
    {'Template': 'T2 – Total Injuries',    'Best Model': best_injuries['model_name'], 'MAE': best_injuries['mae'], 'MAPE%': best_injuries['mape']},
    {'Template': 'T3 – Operator Injuries', 'Best Model': best_op['model_name'],       'MAE': best_op['mae'],       'MAPE%': best_op['mape']},
    {'Template': 'T4 – Rider Injuries',    'Best Model': best_rider['model_name'],    'MAE': best_rider['mae'],    'MAPE%': best_rider['mape']},
])
summary['MAPE%']    = summary['MAPE%'].round(1)
summary['MAE']      = summary['MAE'].round(2)
summary['Reliable'] = summary['MAPE%'].apply(lambda x: '✅ Good' if x < 20 else ('⚠️ Moderate' if x < 35 else '❌ Weak'))

print(summary.to_string(index=False))

  FINAL SUMMARY — Best Model per Template
              Template                     Best Model  MAE  MAPE% Reliable
   T1 – Incident Count SARIMAX(0, 1, 1)×(0, 1, 1, 12) 3.14   11.6   ✅ Good
   T2 – Total Injuries SARIMAX(1, 1, 1)×(0, 1, 1, 12) 5.83   15.8   ✅ Good
T3 – Operator Injuries                 ARIMA(0, 1, 1) 3.08   18.5   ✅ Good
   T4 – Rider Injuries        Prophet(cp=0.1,sp=10.0) 3.22   15.3   ✅ Good
